# Create Dataframes from the below Data 

## usecase -1

The data team has provided two CSV files — customer_data.csv and sales_data.csv. Load them into Spark DataFrames, infer schemas automatically, and print the schema along with the first 5 rows of each.

In [0]:
# create both Customer and transaction drataframes from the given source
cust_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/customer_data.csv",inferSchema=True,header=True)
sales_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/sales_data.csv",inferSchema=True,header=True)

cust_df.show(5)
sales_df.show(5)
cust_df.printSchema()
sales_df.printSchema()


## usecase- 2

The marketing team wants to know the total revenue generated by the 'Clothing' category for each month.

 Revenue = quantity × price.



In [0]:
from pyspark.sql.functions import col,date_format,month,sum
reve_df=sales_df.withColumn("total_revenue",(col("quantity")*col("price"))).filter(col("category")=='Clothing').withColumn("month", month(col("invoice_date")))
reve_df.show(2)
ttal_df=reve_df.groupBy(col("month")).agg(sum("total_revenue").alias("usee by month"))
ttal_df.show()

## usecase 3

Who is the best customer? Find the customer who has spent the most money overall. Include their customer_id and name (from the customer table). Show the top 5.

In [0]:
best_df=cust_df.join(sales_df,['customer_id'],how='inner')
best_df.show(5)

#select customer_id from cust_id order by price desc limit 5

best1_df=best_df.groupBy("customer_id").sum("price").orderBy(col("sum(price)").desc())
best1_df.show(5)

## usecase 4

Generate a customer revenue summary.

 For each customer, show their total revenue and classify them as
  'High Value' (≥ 5000), 
  'Mid Value' (≥ 1000), or 
  'Low Value' (< 1000)

In [0]:
from pyspark.sql.functions import when

revs_df=sales_df.withColumn("total_revenue",(col("quantity")*col("price"))).groupBy(col('customer_id')).agg(sum("total_revenue").alias("total_revenue"))
rev_df=revs_df.withColumn("Value", when(col("total_revenue") >= 5000, 'High Value').when(col("total_revenue") >= 1000, 'Mid Value').otherwise('Low Value'))
rev_df.show()

## usecase 5

Is there a significant spending difference between male and female customers? Compute average transaction value, total revenue, and transaction count by gender.

In [0]:

from pyspark.sql.functions import *
best_df_with_revenue = best_df.withColumn("total_revenue",(col("quantity")*col("price")))
best_df_with_revenue.show()
gender_df = best_df_with_revenue.groupBy("gender").agg(
    avg("price").alias("avg_price"),
    sum("total_revenue").alias("total_revenue"),
    count("*").alias("transaction_count")
)
gender_df.show()

## usecase-6

The finance team wants to know which payment method is most popular for each product category. Show the count of transactions per payment method per category.

In [0]:
counttr_df=best_df.groupBy("payment_method","category").agg(
    count("*").alias("transaction_count"))
counttr_df.show()

## usecase 7

Data integrity check: Are there any transactions with a customer_id that does not exist in the customer master table? Identify and count orphaned transactions.


In [0]:
di_df=cust_df.join(sales_df,['customer_id'],how='leftanti')
di_df.count()


## usecase 8

The marketing team wants to target ads by age group. Bucket customers into: Teens (< 20), Young Adults (20–35), Adults (36–50), Seniors (50+). Which segment generates the most revenue?

In [0]:
age_df = best_df.withColumn("agegroup", 
    when(col("age") < 20, "Teens")
    .when((col("age") >= 20) & (col("age") <= 35), "Young Adults")
    .when((col("age") >= 36) & (col("age") <= 50), "Adults")
    .when(col("age") >= 51, "Seniors")
).withColumn("total_revenue", col("quantity") * col("price"))

revenue_by_age = age_df.groupBy("agegroup").agg(sum("total_revenue").alias("total_revenue")).orderBy(col("total_revenue").desc())

revenue_by_age.show()

## usecase 9

The retention team wants to run a win-back campaign. Identify all customers whose most recent purchase is more than 90 days before the latest invoice date in the dataset.

consider max_date from the table as current_date

In [0]:
from pyspark.sql.functions import max, datediff, lit

#max_date = age_df.agg(max("invoice_date")).first()[0]
max_date = age_df.agg(max("invoice_date")).collect()[0][0]
print(max_date)
result = (
    age_df.groupBy("customer_id")
      .agg(max("invoice_date").alias("last_purchase"))
      .withColumn(
          "days_since_last_purchase",
          datediff(lit(max_date), "last_purchase")
      )
       .withColumn(
          "days_since_last_purchases",
          lit(max_date)
      )
      .filter("days_since_last_purchase > 90")
)

result.select("customer_id").show()

## usecase 10

Create a clean, analysis-ready master dataset by joining customer and transaction tables. Derive revenue, month, day-of-week, and age group. Handle nulls. Persist as a Parquet table.

In [0]:
par_df=age_df.withColumn("month", month(col("invoice_date"))).withColumn("dayofweek", dayofweek(col("invoice_date"))).dropna()
par_df.show()
par_df.write.format("delta").mode("overwrite").saveAsTable("parquetcustomer")


## usecase 11

What is the gender distribution across different product categories

In [0]:
par_gender=par_df.groupBy("category","gender").count().orderBy(col("category"))
par_gender.show()

## usecase 12

What is the total revenue generated in the year 2022

In [0]:
part_df=par_df.withColumn("year", year(col("invoice_date")))
year_df=part_df.groupBy("year").agg(sum("total_revenue").alias("total_revenue"))
year_df.filter(col("year") == 2022).show()